# AG_PRAXIS NB02 — Exploratory Analysis

The inventory told me how many files there are, how many rows, and how the recordings are
organised. It did not tell me anything about what the measurements in those files actually
contain. This notebook reads the values themselves and tries to answer three questions
before any model is built.

Which of the forty-four features carry information, and which are so nearly empty that
they can only ever separate a handful of classes. Which features are near-duplicates of
each other, saying the same thing twice. And which pairs of attacks a single measurement
cannot tell apart, because those are the pairs a classifier will fail on and I would
rather know their names now than read them off a confusion matrix in three notebooks'
time.

There is one more question underneath those, and it is the uncomfortable one. Each attack
class here was recorded in its own session. If a feature's value shifts between two
recordings of the same attack, then part of what that feature measures is the recording
and not the attack, and a model reading it is partly learning which file a row came from.
So for the features that carry the most information about the label, I also look at
whether they hold steady across the recordings of a single class.

Eight and three quarter million rows will not fit in memory all at once, so nothing here
holds the data. Every number is built by reading one file at a time into an accumulator.
The mean, the spread and the skew come from running sums. Everything that depends on rank
rather than on value, which is the median, the mutual information and every pairwise AUC,
comes from a table of how many rows of each class fell in each bin of each feature. That
table is built once and answers all three.

The whole pass is written as one function and run twice. The fast pass does every step on
a stratified sample, so a broken path or a mistake in the arithmetic surfaces in a couple
of minutes rather than most of an hour. The full pass repeats it over every row. The fast
pass is a test of the code and nothing it produces is a measurement of the dataset. The
last cells put the two side by side and print PASS or MISMATCH for every value that cannot
legitimately differ between them, and the ledger entry refuses to describe itself as a
reference run if any of them disagree.

Nothing is trained. The full pass writes `eda_summary.json`, `pairwise_auc.csv` and seven
figures.

The data sits on Drive and the code sits in the repository, so the first block mounts one
and clones the other, and records the commit it is running from. Every number below belongs
to that commit.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Paths and the seed come from `config/base.yaml`, so changing a path is a commit rather than
an edit to a cell that I later forget I made.

The two passes write to two different folders, `NB02_fast` and `NB02`, so neither can
overwrite the other. `NB02` is the full pass and is the one the repository takes.

In [ ]:
import itertools
import json
import random
import time

import numpy as np
import pandas as pd

from src import captures as cap
from src import eda
from src import inventory as inv

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
TRAIN_DIR = Path(CFG["paths"]["train_dir"])
TEST_DIR = Path(CFG["paths"]["test_dir"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])
OUT_DIRS = {"fast": ARTIFACTS / "NB02_fast", "full": ARTIFACTS / "NB02"}

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(
        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "
        "config/base.yaml is wrong. Nothing this notebook writes would survive."
    )

ALL_FILES = sorted(TRAIN_DIR.glob("*.csv")) + sorted(TEST_DIR.glob("*.csv"))
if not ALL_FILES:
    raise FileNotFoundError(f"no CSV files under {TRAIN_DIR} or {TEST_DIR}")

pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

print(f"seed       : {SEED}")
print(f"train dir  : {TRAIN_DIR}   exists={TRAIN_DIR.exists()}")
print(f"test dir   : {TEST_DIR}   exists={TEST_DIR.exists()}")
print(f"files      : {len(ALL_FILES)}")
print(f"fast pass  : {OUT_DIRS['fast']}")
print(f"full pass  : {OUT_DIRS['full']}   <- the one the repository takes")

The column list, the class list and the tier of each class were all settled by the
inventory, so they are read from the file it wrote rather than retyped here. Retyping them
is how two notebooks end up disagreeing about how many classes there are.

One column is dropped. The inventory read every row of every file and found exactly one
column, `Drate`, holding the same value throughout the whole dataset. A column like that
separates nothing from nothing and carries no information by construction, so it goes and
forty-four features remain.

`DHCP` is kept. An earlier screen ran on the opening one percent of each file and reported
that column as constant too, but the head of a file is not the file, and reading all of it
showed `DHCP` moving. That is the general lesson rather than a detail about one column: a
constancy claim taken from a sample is only a claim about the sample, which is why this
notebook takes its column list from the full-scale scan and why nothing below reports a
constancy finding of its own.

In [ ]:
INVENTORY_CANDIDATES = [
    REPO_ROOT / "data" / "processed" / "dataset_inventory.json",
    ARTIFACTS / "NB01" / "dataset_inventory.json",
]
INVENTORY_PATH = next((p for p in INVENTORY_CANDIDATES if p.exists()), None)
if INVENTORY_PATH is None:
    raise FileNotFoundError(
        "dataset_inventory.json not found. NB01 has to have been run, and its output moved "
        f"into data/processed/. Looked in: {[str(p) for p in INVENTORY_CANDIDATES]}"
    )

INVENTORY = json.loads(INVENTORY_PATH.read_text())
if INVENTORY.get("is_fast_pass"):
    raise ValueError(f"{INVENTORY_PATH} was written by NB01's fast pass and is not a result")

ALL_COLUMNS = list(INVENTORY["columns"])
CONSTANT_EVERYWHERE = list(INVENTORY["constant_columns"]["constant_everywhere"])
DROPPED = ["Drate"]
FEATURES = [c for c in ALL_COLUMNS if c not in DROPPED]

CLASS_INFO = INVENTORY["classes"]
CLASSES = sorted(CLASS_INFO)
TIER = {label: CLASS_INFO[label]["tier"] for label in CLASSES}
GROUP6 = {label: CLASS_INFO[label]["group6"] for label in CLASSES}
GROUP6_ORDER = ["Benign", "DDoS", "DoS", "MQTT", "Recon", "Spoofing"]
ROWS_PER_CLASS_NB01 = {label: CLASS_INFO[label]["rows"] for label in CLASSES}
TOTAL_ROWS_NB01 = int(INVENTORY["total_rows"])

print(f"inventory read from {INVENTORY_PATH}")
print(f"  written by  : {INVENTORY['generated_by']} at {INVENTORY['git_sha']} on "
      f"{INVENTORY['generated_on']}")
print(f"  rows scanned: {INVENTORY['rows_scanned']:,}")
print()
print(f"columns in the files            : {len(ALL_COLUMNS)}")
print(f"constant across the whole scan  : {CONSTANT_EVERYWHERE or 'none'}")
print(f"dropped here                    : {DROPPED}")
print(f"features analysed               : {len(FEATURES)}")
print(f"DHCP retained                   : {'DHCP' in FEATURES}")
print(f"classes                         : {len(CLASSES)}")
print(f"tier A / tier B                 : {sum(t == 'A' for t in TIER.values())} / "
      f"{sum(t == 'B' for t in TIER.values())}")
print()

if set(CONSTANT_EVERYWHERE) != set(DROPPED):
    print("The inventory's constant columns are not the ones dropped here.")
    print(f"  inventory found : {CONSTANT_EVERYWHERE}")
    print(f"  dropped         : {DROPPED}")

assert len(FEATURES) == 44, f"expected 44 features after the drop, got {len(FEATURES)}"
assert len(CLASSES) == 19, f"expected 19 classes, got {len(CLASSES)}"
assert all(cap.group_six(c) == GROUP6[c] for c in CLASSES), "6-class map disagrees with NB01"
print("44 features and 19 classes confirmed, and the 6-class map agrees with src/captures.py.")

No model is built here, but the seed is set before anything else runs so that the sample
this notebook draws is the same sample the next time it runs.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
print(f"seeded with {SEED}")

What follows is one function per step, then a driver that calls them in order, then the
cell that runs the whole thing twice. Every step prints its own results and returns them,
so nothing is recomputed later and the comparison at the end has something to compare.

This cell holds the choices the steps share: how many bins, how many rows the sample takes
from each file, and the thresholds at which a feature counts as nearly empty, a pair of
features counts as near-duplicate, and a pair of classes counts as hard to separate. They
sit here rather than inside the steps so that all of them are visible in one place and none
of them is buried in a function.

The four timing features are the whole timing family once `Drate` is gone. The four
contrast features are two from the protocol family and two from the statistical family,
picked to be different shapes: one continuous, one flag, and two summaries of packet size.
The four classes named at the bottom are the ones the published model on this dataset
scores worst on, so it is worth asking separately what carries information about them.

In [ ]:
NBINS = 256
SAMPLE_ROWS_PER_FILE = 10_000

ZERO_THRESHOLD = 0.90
CORR_THRESHOLD = 0.95
AUC_THRESHOLD = 0.90

TOP_MI = 20
TOP_SHIFT = 10

TIMING_FEATURES = ["Duration", "Rate", "Srate", "IAT"]
CONTRAST_FEATURES = ["Header_Length", "TCP", "AVG", "Variance"]
WEAKEST_CLASSES = ["Recon-VulScan", "Recon-OS_Scan", "MQTT-DDoS-Publish_Flood", "Spoofing"]

missing = [f for f in TIMING_FEATURES + CONTRAST_FEATURES if f not in FEATURES]
assert not missing, f"named features not in the column list: {missing}"
missing = [c for c in WEAKEST_CLASSES if c not in CLASSES]
assert not missing, f"named classes not in the class list: {missing}"


def section(title):
    print()
    print("-" * 79)
    print(title)
    print("-" * 79)


def banner(lines):
    print()
    print("#" * 79)
    for line in lines:
        print(f"#  {line:<75}#")
    print("#" * 79)


def merge(results, part):
    """Fold a step's return value into the run, keeping figures from every step."""
    results["figures"] = results.get("figures", []) + list(part.pop("figures", []))
    results.update(part)
    return results


print(f"bins per feature       : {NBINS} quantiles, duplicates collapsed")
print(f"sample rows per file   : {SAMPLE_ROWS_PER_FILE:,}")
print(f"nearly empty above     : {ZERO_THRESHOLD:.0%} of rows zero")
print(f"near-duplicate above   : |r| >= {CORR_THRESHOLD}")
print(f"hard to separate below : AUC {AUC_THRESHOLD:.2f}")
print(f"timing features        : {', '.join(TIMING_FEATURES)}")
print(f"contrast features      : {', '.join(CONTRAST_FEATURES)}")
print(f"weakest classes        : {', '.join(WEAKEST_CLASSES)}")

The figures all share one look, so the styling lives in a single cell rather than being
repeated in seven places. Colour carries the six-class grouping wherever a figure needs it.

Figures drawn by the fast pass get a line at the top of the title saying so. They are drawn
because I want the plotting code exercised in the cheap pass rather than discovering a
broken axis at the end of the long one, and they are labelled because a figure drawn from a
sample looks exactly like a result and is not one.

In [ ]:
import inspect

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# Matplotlib renamed the boxplot orientation argument. Ask rather than guess,
# so that whichever version Colab is on today, the boxes come out horizontal.
HORIZONTAL = (
    {"orientation": "horizontal"}
    if "orientation" in inspect.signature(plt.Axes.boxplot).parameters
    else {"vert": False}
)

INK, MUTED, GRID, RULE = "#0b0b0b", "#52514e", "#e6e5e1", "#c9c8c3"
GROUP_COLOUR = {
    "Benign": "#3f6fb0",
    "DDoS": "#d1622b",
    "DoS": "#e0a02c",
    "MQTT": "#4b9c76",
    "Recon": "#8a5fa8",
    "Spoofing": "#b03a4a",
}
FLAG = "#b03a4a"


def _style(ax, axis="x"):
    ax.grid(axis=axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(RULE)
    ax.tick_params(colors=MUTED, length=0)
    for label in ax.get_yticklabels():
        label.set_color(INK)


def _title(text, fast):
    return ("FAST PASS, drawn from a sample, not a result\n" + text) if fast else text


def _group_legend(target, groups, **kwargs):
    """A 6-class colour key, on an axis or on the figure itself."""
    handles = [
        plt.Rectangle((0, 0), 1, 1, color=GROUP_COLOUR[g], label=g)
        for g in GROUP6_ORDER
        if g in set(groups)
    ]
    legend = target.legend(handles=handles, frameon=False, fontsize=8, **kwargs)
    for text in legend.get_texts():
        text.set_color(MUTED)


def _needs_log(values):
    """Whether a spread is wide enough that a linear axis hides most of it."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v) & (v > 0)]
    if v.size < 8:
        return False, 1.0
    lo, hi = np.percentile(v, [1, 99])
    lo = max(lo, float(v.min()), 1e-12)
    return bool(hi / lo > 100), float(lo)


def _save(fig, path):
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"wrote {path}")
    plt.show()
    plt.close(fig)
    return path


print("figure style set, 300 dpi on save")

Two things have to be read before any step can run.

The first is the reference sample: the opening rows of every file, capped at ten thousand
each. One file holds exactly one class, so taking the same number of rows from every file
puts every class and every recording in the sample, including the class with nine hundred
rows in total. What it does not do is preserve the relative sizes of the classes, so it is
stratified in the sense of covering everything and not in the sense of being a scale model
of the dataset. It is also the head of each file rather than a random draw, because
stopping the reader early is the only version of this that is actually cheap, and rows
inside a file are in the order the extractor wrote them. That makes the sample a contiguous
slice of each session rather than a sample of it.

Both passes read this sample and read it identically. Two steps are computed from it in
both passes, and it also fixes the bin edges, so the fast and the full pass put their rows
into exactly the same bins and differ only in how many rows they put there.

The second is the bin edges themselves. Each feature gets boundaries at two hundred and
fifty-six evenly spaced quantiles of the sample, with duplicates collapsed. Quantiles
rather than an even grid, so that the bins land where the values are. Collapsing duplicates
matters more than it sounds: a column that is zero in ninety-five percent of rows would
otherwise get two hundred boundaries all sitting on zero, and instead it gets a handful of
bins that fall where it actually varies. A column taking a handful of distinct values gets
a boundary at each of them and is not approximated at all.

In [ ]:
def read_features(path, *, nrows=None):
    """Read a capture file, or its opening rows, and keep the analysed columns."""
    frame = pd.read_csv(path, nrows=nrows)
    absent = [c for c in FEATURES if c not in frame.columns]
    if absent:
        raise KeyError(f"{Path(path).name} is missing {absent}")
    frame = frame[FEATURES]
    wrong = {c: str(t) for c, t in frame.dtypes.items() if not pd.api.types.is_numeric_dtype(t)}
    assert not wrong, f"{Path(path).name} has non-numeric feature columns: {wrong}"
    return frame


def build_reference_sample(all_files):
    section("The reference sample, read the same way in both passes")

    started = time.time()
    frames = []
    for path in all_files:
        meta = cap.parse_capture(path.name)
        frame = read_features(path, nrows=SAMPLE_ROWS_PER_FILE).astype("float32")
        frame["file"] = path.name
        frame["label"] = meta["label"]
        frame["capture_id"] = meta["capture_id"]
        frame["group6"] = cap.group_six(meta["label"])
        frames.append(frame)

    # The four label columns stay as plain strings. Making them categorical
    # would save a little memory and would quietly turn anything derived from
    # them into a categorical as well, because pandas carries the dtype through
    # groupby and through map, and a categorical of counts will not divide. It
    # does that only when the mapped values happen to be distinct, so code
    # written that way works on data where two classes are the same size and
    # fails on data where they are not, which is the worst way to find out.
    sample = pd.concat(frames, ignore_index=True)
    del frames

    counts = sample.groupby("label").size()
    per_class = pd.DataFrame(
        {
            "label": [str(label) for label in counts.index],
            "sample_rows": counts.to_numpy(dtype="int64"),
        }
    )
    per_class["tier"] = per_class["label"].map(TIER)
    per_class["dataset_rows"] = per_class["label"].map(ROWS_PER_CLASS_NB01).astype("int64")

    assert per_class[["sample_rows", "dataset_rows"]].dtypes.map(
        lambda d: pd.api.types.is_numeric_dtype(d)).all(), \
        f"non-numeric count columns: {per_class.dtypes.to_dict()}"

    per_class["share_of_class"] = (per_class["sample_rows"] / per_class["dataset_rows"]).round(4)
    per_class = per_class.sort_values("sample_rows", ascending=False).reset_index(drop=True)

    print(f"read {len(all_files)} files in {time.time() - started:.0f}s, "
          f"at most {SAMPLE_ROWS_PER_FILE:,} rows each")
    print(f"sample : {len(sample):,} rows, {len(FEATURES)} features")
    print(f"         {100 * len(sample) / TOTAL_ROWS_NB01:.2f}% of the {TOTAL_ROWS_NB01:,} "
          "rows in the dataset")
    print()
    print(per_class.to_string(index=False))
    print()
    print("Every class is present. The proportions are not the dataset's, which is why the")
    print("steps that depend on how common a class is are run again over every row.")

    return {
        "sample": sample,
        "sample_rows": int(len(sample)),
        "sample_rows_per_class": {r["label"]: int(r["sample_rows"]) for _, r in per_class.iterrows()},
        "sample_read_s": time.time() - started,
    }


def build_bin_edges(sample):
    section(f"Bin edges, up to {NBINS} quantiles per feature, taken from the sample")

    edges = {f: eda.quantile_edges(sample[f].to_numpy(dtype=np.float64), NBINS) for f in FEATURES}
    widths = pd.DataFrame(
        {"feature": FEATURES, "bins": [len(edges[f]) + 1 for f in FEATURES]}
    ).sort_values("bins")

    print("features with the fewest bins, which are the ones taking few distinct values")
    print(widths.head(12).to_string(index=False))
    print()
    print(f"bins per feature : min {widths['bins'].min()}, median "
          f"{int(widths['bins'].median())}, max {widths['bins'].max()}")
    print()
    print("A feature with only a handful of bins takes only a handful of distinct values,")
    print("and the boundaries land on them, so nothing about it is being approximated. A")
    print(f"feature at the maximum is read to a resolution of one part in {NBINS}, which is")
    print("finer than any threshold used below.")

    return {"bin_edges": edges, "bins_per_feature": {f: len(edges[f]) + 1 for f in FEATURES}}

Then the reading pass, which is the only place data is read in bulk and the only step whose
cost depends on which pass is running. The full pass reads every row of every file. The
fast pass reads the same opening rows the sample took.

Three accumulators are filled at once, because reading the files is the expensive part and
reading them three times would be three times the cost for no extra information. The first
holds running sums per feature, enough for the mean, the standard deviation and the skew,
along with the smallest and largest value seen and a count of the values that were exactly
zero. The second is the table of bin counts per class and feature. The third is the same
kind of running sums again but kept separately for each of the fifty-seven recordings,
which is what the question about recording shift needs.

The fast pass reads its rows through this same function rather than reusing the sample it
already has in memory. That is a little wasteful and it is deliberate: nothing is computed
a different way in fast mode, so any value the two passes both produce was produced by
exactly the same code, and a disagreement between them can only be a bug.

In [ ]:
def reading_pass(all_files, bin_edges, sample, *, fast):
    scope = "the sample's rows" if fast else "every row of every file"
    section(f"Reading {scope}")

    offsets = {f: float(sample[f].mean()) for f in FEATURES}
    moments = eda.Moments(FEATURES, offsets)
    counts = eda.BinnedCounts(CLASSES, FEATURES, bin_edges)
    per_capture = {}
    rows_read = 0
    started = time.time()

    for i, path in enumerate(all_files, start=1):
        meta = cap.parse_capture(path.name)
        values = read_features(
            path, nrows=SAMPLE_ROWS_PER_FILE if fast else None
        ).to_numpy(dtype=np.float64)

        moments.update(values)
        counts.add(meta["label"], values)
        record = per_capture.setdefault(
            meta["capture_id"],
            {"label": meta["label"], "moments": eda.Moments(FEATURES, offsets)},
        )
        record["moments"].update(values)
        rows_read += len(values)

        if i % 12 == 0 or i == len(all_files):
            print(f"  {i:>2}/{len(all_files)} files, {rows_read:,} rows, "
                  f"{time.time() - started:.0f}s")
        del values

    print()
    print(f"rows read   : {rows_read:,}")
    print(f"recordings  : {len(per_capture)}")
    print(f"of the dataset's {TOTAL_ROWS_NB01:,} rows : {100 * rows_read / TOTAL_ROWS_NB01:.2f}%")

    seen = counts.rows_per_class
    if not fast:
        expected = np.array([ROWS_PER_CLASS_NB01[c] for c in CLASSES])
        disagree = [
            (c, int(s), int(e)) for c, s, e in zip(CLASSES, seen, expected) if s != e
        ]
        if disagree:
            print()
            print("The rows read per class do not match the counts the inventory recorded:")
            for label, got, want in disagree:
                print(f"  {label:<28} read {got:,}, inventory says {want:,}")
        else:
            print("Every class holds exactly the number of rows the inventory recorded.")
    else:
        print("Class sizes here are the sample's, not the dataset's.")

    nonfinite = int(moments.nonfinite.sum())
    print(f"values that were not finite : {nonfinite:,}"
          + ("" if nonfinite == 0 else "   <- these are excluded from every statistic below"))

    return {
        "moments": moments,
        "counts": counts,
        "per_capture": per_capture,
        "rows_read": int(rows_read),
        "rows_per_class_read": {c: int(n) for c, n in zip(CLASSES, seen)},
        "nonfinite_values": nonfinite,
        "reading_pass_s": time.time() - started,
    }

The first thing to look at is what each feature is, one at a time: where it sits, how far
it spreads, how far it runs, and how lopsided it is. The mean, the standard deviation, the
smallest and largest value and the skew all come out of the running sums. The median comes
off the bin counts, so it is accurate to the width of one bin rather than exact, which for
a column with two hundred bins means the last figure is not to be trusted and for a column
with three bins is not an approximation at all.

The column I care about most is the last one, the share of rows in which the feature is
exactly zero. A feature that is zero in ninety-nine percent of rows is not describing the
traffic in general. It is a marker that fires on one kind of traffic and is silent
otherwise, and the only thing it can ever do is separate the few classes where it fires
from everything else. That is not useless, and for a rare class it can be the most valuable
thing in the file, but it does mean the feature cannot contribute to telling the other
seventeen classes apart, and it means any average computed over it is dominated by rows
where nothing happened.

The skew is worth reading next to it. A feature that is nearly always zero and occasionally
enormous will show a skew in the hundreds, which is a warning about scaling rather than
about information: a standardiser fitted on a column like that produces values that are
almost all slightly negative and a handful that are in the thousands.

In [ ]:
def summarise_features(moments, counts, *, fast, out_dir):
    section("What each of the 44 features looks like")

    summary = moments.frame()
    summary["median"] = [counts.quantile(f, 0.5) for f in FEATURES]
    summary["bins"] = [int(counts.n_bins[FEATURES.index(f)]) for f in FEATURES]
    summary = summary[
        ["feature", "rows", "mean", "std", "min", "median", "max", "skew", "zero_share", "bins"]
    ]

    shown = summary.copy()
    for column in ("mean", "std", "min", "median", "max"):
        shown[column] = shown[column].map(lambda v: f"{v:,.4g}")
    shown["skew"] = shown["skew"].map(lambda v: f"{v:,.2f}")
    shown["zero_share"] = shown["zero_share"].map(lambda v: f"{v:.4f}")
    shown["rows"] = shown["rows"].map(lambda v: f"{v:,}")
    print(shown.to_string(index=False))
    print()

    mostly_zero = summary.loc[summary["zero_share"] > ZERO_THRESHOLD].sort_values(
        "zero_share", ascending=False
    )
    print("=" * 79)
    print(f"Features that are zero in more than {ZERO_THRESHOLD:.0%} of rows")
    print("=" * 79)
    if len(mostly_zero):
        print(
            mostly_zero[["feature", "zero_share", "max", "skew"]]
            .assign(
                zero_share=lambda f: f["zero_share"].map(lambda v: f"{v:.4f}"),
                max=lambda f: f["max"].map(lambda v: f"{v:,.4g}"),
                skew=lambda f: f["skew"].map(lambda v: f"{v:,.1f}"),
            )
            .to_string(index=False)
        )
        print()
        print(f"{len(mostly_zero)} of {len(FEATURES)} features are silent in more than")
        print(f"{ZERO_THRESHOLD:.0%} of rows. Each of them can only separate the classes where it")
        print("is not zero from everything else. It cannot help tell the remaining classes")
        print("apart, and its mean and standard deviation are mostly a statement about how")
        print("often it is zero rather than about what it measures when it fires.")
    else:
        print(f"None. Every feature is non-zero in at least {1 - ZERO_THRESHOLD:.0%} of rows.")
    print()

    all_zero = summary.loc[summary["zero_share"] >= 1.0, "feature"].tolist()
    if all_zero:
        print(f"Zero in every row read : {all_zero}. These carry nothing at all.")
    else:
        print("No feature is zero in every row, so none of them is dead on this reading.")

    order = summary.sort_values("zero_share")
    fig, ax = plt.subplots(figsize=(9, 10))
    colours = [FLAG if v > ZERO_THRESHOLD else MUTED for v in order["zero_share"]]
    bars = ax.barh(order["feature"], order["zero_share"], color=colours, height=0.7)
    ax.bar_label(
        bars, labels=[f"{v:.3f}" for v in order["zero_share"]], padding=3, color=MUTED, fontsize=7
    )
    ax.axvline(ZERO_THRESHOLD, color=FLAG, linewidth=1, linestyle="--")
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("share of rows where the feature is exactly zero", color=MUTED, fontsize=9)
    ax.set_title(
        _title(
            f"How often each feature is zero, {len(mostly_zero)} of {len(FEATURES)} above "
            f"{ZERO_THRESHOLD:.0%}",
            fast,
        ),
        color=INK,
        fontsize=11,
        loc="left",
        pad=12,
    )
    _style(ax)
    ax.tick_params(axis="y", labelsize=8)
    path = _save(fig, out_dir / "NB02_zero_share.png")

    return {
        "feature_summary": summary,
        "mostly_zero": mostly_zero["feature"].tolist(),
        "all_zero": all_zero,
        "figures": [path],
    }

Next, whether the features are saying the same thing as each other. Correlation between all
forty-four, drawn as a heatmap with the features ordered so that the ones behaving alike
sit together. The ordering comes from clustering on one minus the absolute correlation, so
a pair carrying the same information ends up adjacent whichever sign it carries it with,
and blocks of near-duplicates show up as squares on the diagonal.

Two columns saying the same thing do not break a classifier. It will use one, or split the
difference, and score about the same either way. The problem arrives later, when the model
is explained. An attribution method distributes credit among the inputs, so if two features
carry the same signal the credit is shared between them and each looks half as important as
the thing it measures actually is. A feature that would have ranked first can end up
outside the top five with no change in what the model learned, purely because it has a
twin. Since this project ends by reading feature attributions and turning them into threat
records, I need to know which features are twins before I read a ranking off them.

This step runs on the reference sample and is one of the two that run identically in both
passes.

In [ ]:
def correlate_features(sample, *, fast, out_dir):
    section("Which features are saying the same thing")

    values = sample[FEATURES].to_numpy(dtype=np.float64)
    with np.errstate(invalid="ignore", divide="ignore"):
        matrix = np.corrcoef(values, rowvar=False)
    corr = pd.DataFrame(matrix, index=FEATURES, columns=FEATURES)
    del values

    undefined = [f for f in FEATURES if not np.isfinite(corr[f].to_numpy()).any()]
    if undefined:
        print("Correlation is undefined for these features because they do not vary in the")
        print(f"sample: {undefined}")
        print()

    pairs = eda.high_correlation_pairs(corr, CORR_THRESHOLD)
    order = eda.cluster_order(corr)

    print("=" * 79)
    print(f"Feature pairs correlating above {CORR_THRESHOLD}")
    print("=" * 79)
    if len(pairs):
        print(pairs.assign(r=lambda f: f["r"].map(lambda v: f"{v: .4f}")).to_string(index=False))
        print()
        involved = len(set(pairs["feature_a"]) | set(pairs["feature_b"]))
        print(f"{len(pairs)} pairs, involving {involved} of the {len(FEATURES)} features.")
        print()
        print("These matter when the model is explained rather than when it is fitted. Credit")
        print("for a signal carried by two columns is split between them, so each looks about")
        print("half as important as the thing it measures. Any ranking read off attributions")
        print("has to be read with this list next to it.")
    else:
        print(f"None. No two features correlate above {CORR_THRESHOLD}.")
    print()

    print("clustered order, related features adjacent")
    print("  " + ", ".join(order))

    ordered = corr.loc[order, order]
    fig, ax = plt.subplots(figsize=(12, 10.5))
    image = ax.imshow(
        ordered.to_numpy(),
        cmap="RdBu_r",
        norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1),
        interpolation="nearest",
    )
    ax.set_xticks(range(len(order)), order, rotation=90, fontsize=7, color=MUTED)
    ax.set_yticks(range(len(order)), order, fontsize=7, color=INK)
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    flagged = {(a, b) for a, b in zip(pairs["feature_a"], pairs["feature_b"])}
    for i, a in enumerate(order):
        for j, b in enumerate(order):
            if (a, b) in flagged or (b, a) in flagged:
                ax.add_patch(
                    plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor=INK, linewidth=0.6)
                )
    bar = fig.colorbar(image, ax=ax, fraction=0.035, pad=0.02)
    bar.outline.set_visible(False)
    bar.ax.tick_params(colors=MUTED, length=0)
    bar.set_label("Pearson r", color=MUTED, fontsize=9)
    ax.set_title(
        _title(
            f"Correlation between all {len(FEATURES)} features, clustered. "
            f"{len(pairs)} pairs above {CORR_THRESHOLD} are outlined",
            fast,
        ),
        color=INK,
        fontsize=11,
        loc="left",
        pad=12,
    )
    path = _save(fig, out_dir / "NB02_correlation_heatmap.png")

    return {
        "correlation": corr,
        "correlation_order": order,
        "high_correlation_pairs": pairs,
        "correlation_undefined": undefined,
        "figures": [path],
    }

Correlation says how features move with each other. It says nothing about whether they move
differently for different attacks, which is the only thing that makes a feature useful. So
the next step splits each feature by the six-class grouping and looks at the shape.

The four timing features go first, because they are the ones I am most suspicious of. A
scan and a flood differ in how fast packets arrive, so timing separating the classes is
expected. Timing separating them very cleanly is a different matter, because the rate at
which packets arrive also depends on the machine, the link and the moment the recording was
made, and those are properties of the session rather than of the attack. This step cannot
tell those two explanations apart. It can show me how much of the separation is sitting in
those four columns, which is what makes the later question about recordings worth asking.

Then four features from the protocol and statistical families for contrast. A feature that
looks the same in all six panels tells the model nothing at this level, however healthy its
summary statistics looked.

Boxes rather than histograms, with whiskers at the fifth and ninety-fifth percentiles, so
that six distributions fit on one axis. Where a feature runs over more than two orders of
magnitude the axis is symmetric log, which keeps the zeros visible instead of throwing them
off the end. A feature taking only two values gets a bar of how often it is set instead,
because a box plot of a flag is a flat line and says nothing.

This step also runs on the reference sample, so it too is identical in both passes.

In [ ]:
def plot_distributions(sample, *, fast, out_dir):
    section("How each feature is distributed across the six classes")

    present = set(map(str, sample["group6"].unique()))
    groups = [g for g in GROUP6_ORDER if g in present]
    masks = {g: (sample["group6"] == g).to_numpy() for g in groups}
    print("rows per 6-class group in the sample: "
          + ", ".join(f"{g} {int(masks[g].sum()):,}" for g in groups))
    print()

    rows = []
    figures = []

    def panel(ax, feature):
        column = sample[feature].to_numpy(dtype=np.float64)
        data = [column[masks[g]] for g in groups]
        distinct = np.unique(np.concatenate(data))
        binary = distinct.size <= 2

        if binary:
            shares = [float((d != 0).mean()) if d.size else np.nan for d in data]
            ax.barh(
                range(len(groups)),
                shares,
                color=[GROUP_COLOUR[g] for g in groups],
                height=0.6,
            )
            ax.set_xlim(0, 1.05)
            ax.set_xlabel("share of rows set", color=MUTED, fontsize=8)
            note = f"takes {distinct.size} distinct values, shown as a share"
        else:
            box = ax.boxplot(
                data,
                positions=np.arange(len(groups)),
                whis=(5, 95),
                **HORIZONTAL,
                showfliers=False,
                widths=0.6,
                patch_artist=True,
                medianprops={"color": INK, "linewidth": 1.2},
                whiskerprops={"color": RULE},
                capprops={"color": RULE},
                boxprops={"edgecolor": RULE, "linewidth": 0.8},
            )
            for patch, g in zip(box["boxes"], groups):
                patch.set_facecolor(GROUP_COLOUR[g])
                patch.set_alpha(0.75)
            needs_log, linthresh = _needs_log(np.concatenate(data))
            if needs_log:
                ax.set_xscale("symlog", linthresh=linthresh)
            ax.set_xlabel("value" + (", symmetric log" if needs_log else ""), color=MUTED, fontsize=8)
            note = "symmetric log axis" if needs_log else "linear axis"

        ax.set_yticks(range(len(groups)), groups, fontsize=8)
        ax.set_title(f"{feature}   ({note})", color=INK, fontsize=10, loc="left", pad=8)
        _style(ax)

        for g, d in zip(groups, data):
            rows.append(
                {
                    "feature": feature,
                    "group6": g,
                    "median": float(np.median(d)) if d.size else np.nan,
                    "p05": float(np.percentile(d, 5)) if d.size else np.nan,
                    "p95": float(np.percentile(d, 95)) if d.size else np.nan,
                    "zero_share": float((d == 0).mean()) if d.size else np.nan,
                    "binary": bool(binary),
                }
            )

    for name, features, title in (
        ("timing", TIMING_FEATURES, "The four timing features, split by 6-class group"),
        ("contrast", CONTRAST_FEATURES, "Two protocol and two statistical features, for contrast"),
    ):
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        for ax, feature in zip(axes.ravel(), features):
            panel(ax, feature)
        fig.suptitle(_title(title, fast), color=INK, fontsize=12, x=0.09, ha="left", y=0.99)
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        figures.append(_save(fig, out_dir / f"NB02_distributions_{name}.png"))

    table = pd.DataFrame(rows)
    print("medians by group, the numbers behind the panels")
    wide = table.pivot(index="feature", columns="group6", values="median")
    wide = wide.reindex(index=TIMING_FEATURES + CONTRAST_FEATURES, columns=groups)
    print(wide.to_string(float_format=lambda v: f"{v:,.4g}"))
    print()
    flat = [
        f
        for f in TIMING_FEATURES + CONTRAST_FEATURES
        if wide.loc[f].nunique(dropna=True) <= 1
    ]
    print(f"features whose median is the same in all six groups : {flat or 'none'}")
    print("A feature like that is not separating anything at this level of grouping. It may")
    print("still separate two classes inside one group, which is the question step 7 asks.")

    return {"distribution_summary": table, "figures": figures}

Now a number for how much each feature tells you about the label. Mutual information: how
much shorter the description of a row's class gets once you know which bin the feature
falls in. Zero means the bin says nothing. The ceiling is the entropy of the class
variable, which for nineteen equally sized classes would be the log of nineteen, about 2.94
nats, and is lower than that here because the classes are nowhere near equally sized.

Both variables are discrete once the feature is binned, so this is the definition summed
over a table rather than an estimate with a bandwidth in it. That matters at this scale.
The usual nearest-neighbour estimator would have to search eight million points forty-four
times; the table is already built and the sum takes a fraction of a second.

Then the same ranking again over four classes only: the four the published model on this
dataset effectively fails to detect. The reason for computing it separately is that mutual
information against all nineteen is dominated by the classes with the most rows, and the
four classes I care about hold a small fraction of the dataset between them. A feature can
rank first overall by separating the two largest floods from everything else while saying
nothing whatsoever about a reconnaissance scan. The two rankings are printed side by side
and the features that move furthest between them are listed, because those are the features
the rare classes depend on and the overall ranking hides.

In [ ]:
def rank_by_information(counts, *, fast, out_dir):
    section("How much each feature tells you about the label")

    mi_all = eda.mutual_information(counts.counts)
    ceiling_all = eda.label_entropy(counts.counts)

    weak = counts.subset(WEAKEST_CLASSES)
    mi_weak = eda.mutual_information(weak)
    ceiling_weak = eda.label_entropy(weak)
    weak_rows = {c: int(n) for c, n in zip(WEAKEST_CLASSES, weak[:, 0, :].sum(axis=1))}

    ranking = pd.DataFrame({"feature": FEATURES, "mi_19": mi_all, "mi_weak4": mi_weak})
    ranking["rank_19"] = ranking["mi_19"].rank(ascending=False, method="min").astype(int)
    ranking["rank_weak4"] = ranking["mi_weak4"].rank(ascending=False, method="min").astype(int)
    ranking["rank_change"] = ranking["rank_19"] - ranking["rank_weak4"]
    ranking = ranking.sort_values("mi_19", ascending=False).reset_index(drop=True)

    def show(frame):
        return frame.assign(
            mi_19=lambda f: f["mi_19"].map(lambda v: f"{v:.4f}"),
            mi_weak4=lambda f: f["mi_weak4"].map(lambda v: f"{v:.4f}"),
        )[["feature", "mi_19", "rank_19", "mi_weak4", "rank_weak4", "rank_change"]].to_string(
            index=False
        )

    print(f"against all {len(CLASSES)} classes, ceiling {ceiling_all:.4f} nats")
    print(show(ranking.head(TOP_MI)))
    print()
    print(f"against the four weakest classes only, ceiling {ceiling_weak:.4f} nats")
    print("rows behind that ranking : " + ", ".join(f"{k} {v:,}" for k, v in weak_rows.items()))
    print(show(ranking.sort_values("mi_weak4", ascending=False).head(TOP_MI)))
    print()

    top_all = set(ranking.head(TOP_MI)["feature"])
    top_weak = set(ranking.sort_values("mi_weak4", ascending=False).head(TOP_MI)["feature"])
    print("=" * 79)
    print("Where the two rankings disagree")
    print("=" * 79)
    print(f"in the top {TOP_MI} for all {len(CLASSES)} classes but not for the weakest four:")
    print("  " + (", ".join(sorted(top_all - top_weak)) or "nothing"))
    print(f"in the top {TOP_MI} for the weakest four but not for all {len(CLASSES)}:")
    print("  " + (", ".join(sorted(top_weak - top_all)) or "nothing"))
    print()

    movers = ranking.reindex(ranking["rank_change"].abs().sort_values(ascending=False).index)
    movers = movers.head(10)
    print("the ten features that move furthest between the two rankings")
    print(
        movers.assign(
            mi_19=lambda f: f["mi_19"].map(lambda v: f"{v:.4f}"),
            mi_weak4=lambda f: f["mi_weak4"].map(lambda v: f"{v:.4f}"),
        )[["feature", "mi_19", "rank_19", "mi_weak4", "rank_weak4", "rank_change"]].to_string(
            index=False
        )
    )
    print()
    print("A positive change means the feature is more useful for the four hard classes than")
    print("its overall rank suggests. Those are the ones that would be lost by selecting")
    print("features on the overall ranking alone, and they are exactly the features the")
    print("classes that currently go undetected depend on.")

    fig, axes = plt.subplots(1, 2, figsize=(13, 7.5), sharex=False)
    for ax, column, ceiling, label in (
        (axes[0], "mi_19", ceiling_all, f"all {len(CLASSES)} classes"),
        (axes[1], "mi_weak4", ceiling_weak, "the four weakest classes"),
    ):
        top = ranking.sort_values(column, ascending=False).head(TOP_MI).iloc[::-1]
        other = top_weak if column == "mi_19" else top_all
        colours = [MUTED if f in other else FLAG for f in top["feature"]]
        bars = ax.barh(top["feature"], top[column], color=colours, height=0.7)
        ax.bar_label(bars, labels=[f"{v:.3f}" for v in top[column]], padding=3,
                     color=MUTED, fontsize=7)
        ax.axvline(ceiling, color=RULE, linewidth=1, linestyle="--")
        ax.set_xlim(0, max(top[column].max(), ceiling) * 1.18)
        ax.set_xlabel(f"nats, ceiling {ceiling:.3f}", color=MUTED, fontsize=9)
        ax.set_title(f"Top {TOP_MI} against {label}", color=INK, fontsize=10, loc="left", pad=8)
        _style(ax)
        ax.tick_params(axis="y", labelsize=8)
    fig.suptitle(
        _title(
            "Mutual information per feature. Marked bars are in this top "
            f"{TOP_MI} but not in the other one",
            fast,
        ),
        color=INK,
        fontsize=12,
        x=0.06,
        ha="left",
        y=0.99,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    path = _save(fig, out_dir / "NB02_mutual_information.png")

    return {
        "mi_ranking": ranking,
        "mi_ceiling_19": float(ceiling_all),
        "mi_ceiling_weak4": float(ceiling_weak),
        "weak_class_rows": weak_rows,
        "figures": [path],
    }

The step above says which features carry the most information about the label. This one
asks where that information comes from.

Each attack class in this dataset was recorded in its own session, and eight of the
nineteen classes were recorded more than once. For those eight I can take a feature and
compare its average across the separate recordings of the same attack. The attack is the
same in all of them, so anything that shifts between them is not the attack. It is the
machine, the link, the load, or the time of day.

The plot shows, for each of the ten features that carry the most information, the mean per
recording with a bar of one standard deviation, with the recordings grouped by class. What
I want to see inside a class is a flat line: the same attack measured the same way each
time. What would worry me is a feature whose recordings sit at clearly different heights
within one class, because the model can then read the recording off that feature, and since
each recording holds one class, reading the recording is most of the way to reading the
label without having learned anything about attacks at all.

The number underneath the plot is the spread between recordings divided by the spread
inside them, computed within a class and then summarised across the eight classes that have
more than one recording. Near zero means the recordings agree. Near or above one means the
gap between two recordings of the same attack is as large as the variation inside a single
recording, which is a feature describing the session.

Eleven classes were recorded once, so for those there is nothing to compare and the
question cannot be asked at all. They are drawn as a single point and that is a limit of
the dataset rather than a result.

In [ ]:
def measure_recording_shift(per_capture, ranking, *, fast, out_dir):
    section("Does a feature shift between recordings of the same attack")

    top = ranking.head(TOP_SHIFT)["feature"].tolist()
    print(f"the {TOP_SHIFT} features carrying the most information: {', '.join(top)}")
    print()

    rows = []
    for capture_id, record in per_capture.items():
        moments = record["moments"]
        frame = moments.frame().set_index("feature")
        for feature in top:
            rows.append(
                {
                    "capture_id": capture_id,
                    "label": record["label"],
                    "group6": GROUP6[record["label"]],
                    "tier": TIER[record["label"]],
                    "feature": feature,
                    "n": int(frame.loc[feature, "rows"]),
                    "mean": float(frame.loc[feature, "mean"]),
                    "std": float(frame.loc[feature, "std"]),
                }
            )
    per_recording = pd.DataFrame(rows)
    assert per_recording[["n", "mean", "std"]].dtypes.map(
        lambda d: pd.api.types.is_numeric_dtype(d)).all(), \
        f"non-numeric count columns: {per_recording.dtypes.to_dict()}"

    counts_per_class = per_recording.groupby("label")["capture_id"].nunique()
    multi = sorted(counts_per_class[counts_per_class > 1].index)
    single = sorted(counts_per_class[counts_per_class <= 1].index)
    print(f"classes with more than one recording : {len(multi)}   {', '.join(multi)}")
    print(f"classes with one recording           : {len(single)}")
    print("For the second group the question cannot be asked. There is nothing to compare a")
    print("recording against.")
    print()

    shift_rows = []
    for feature in top:
        block = per_recording[per_recording["feature"] == feature]
        ratios = {}
        for label in multi:
            part = block[block["label"] == label]
            n = part["n"].to_numpy(dtype=float)
            if n.sum() <= 0 or len(part) < 2:
                continue
            means = part["mean"].to_numpy(dtype=float)
            grand = np.average(means, weights=n)
            between = np.average((means - grand) ** 2, weights=n)
            within = np.average(part["std"].to_numpy(dtype=float) ** 2, weights=n)
            if within > 0:
                ratios[label] = float(np.sqrt(between / within))
        if ratios:
            worst = max(ratios, key=ratios.get)
            shift_rows.append(
                {
                    "feature": feature,
                    "classes_measured": len(ratios),
                    "median_ratio": float(np.median(list(ratios.values()))),
                    "max_ratio": float(ratios[worst]),
                    "worst_class": worst,
                    "per_class": {k: round(v, 4) for k, v in ratios.items()},
                }
            )

    shift = pd.DataFrame(shift_rows).sort_values("max_ratio", ascending=False)
    print("spread between recordings divided by spread inside them, within a class")
    print(
        shift[["feature", "classes_measured", "median_ratio", "max_ratio", "worst_class"]]
        .assign(
            median_ratio=lambda f: f["median_ratio"].map(lambda v: f"{v:.3f}"),
            max_ratio=lambda f: f["max_ratio"].map(lambda v: f"{v:.3f}"),
        )
        .to_string(index=False)
    )
    print()
    large = shift[shift["max_ratio"] >= 1.0]
    if len(large):
        print(f"{len(large)} of these features have at least one class where the recordings sit")
        print("further apart than the rows inside a single recording do:")
        for _, row in large.iterrows():
            print(f"  {row['feature']:<16} {row['max_ratio']:.2f} on {row['worst_class']}")
        print()
        print("For those features part of what is being measured is the recording session.")
        print("A model reading them can recover which file a row came from, and because each")
        print("recording holds exactly one class, that is most of the way to the label.")
    else:
        print("No feature reaches a ratio of one. On this measure the recordings of a class")
        print("agree with each other more closely than the rows inside a recording do.")
    print()
    print("This does not clear the features. The ratio compares means, so a feature whose")
    print("mean holds steady while its shape changes between recordings would pass it.")

    ordered_classes = sorted(CLASSES, key=lambda c: (GROUP6_ORDER.index(GROUP6[c]), c))
    layout = (
        per_recording[["capture_id", "label"]]
        .drop_duplicates()
        .assign(order=lambda f: f["label"].map(ordered_classes.index))
        .sort_values(["order", "capture_id"])
        .reset_index(drop=True)
    )
    position = {c: i for i, c in enumerate(layout["capture_id"])}
    boundaries = {}
    for label in ordered_classes:
        here = [position[c] for c, l in zip(layout["capture_id"], layout["label"]) if l == label]
        if here:
            boundaries[label] = (min(here), max(here))

    fig, axes = plt.subplots(5, 2, figsize=(14, 16), sharex=True)
    for ax, feature in zip(axes.ravel(), top):
        block = per_recording[per_recording["feature"] == feature]
        for label in ordered_classes:
            part = block[block["label"] == label]
            if not len(part):
                continue
            x = [position[c] for c in part["capture_id"]]
            ax.errorbar(
                x,
                part["mean"],
                yerr=part["std"],
                fmt="o",
                markersize=4,
                elinewidth=1,
                capsize=2,
                color=GROUP_COLOUR[GROUP6[label]],
                ecolor=GROUP_COLOUR[GROUP6[label]],
                alpha=0.9,
            )
        needs_log, linthresh = _needs_log(np.abs(block["mean"].to_numpy()))
        if needs_log:
            ax.set_yscale("symlog", linthresh=linthresh)
        for k, (lo, hi) in enumerate(boundaries.values()):
            if k % 2 == 0:
                ax.axvspan(lo - 0.5, hi + 0.5, color=GRID, alpha=0.45, linewidth=0)
        ax.set_title(feature, color=INK, fontsize=10, loc="left", pad=6)
        ax.set_ylabel("mean per recording", color=MUTED, fontsize=8)
        _style(ax, axis="y")

    for ax in axes[-1]:
        ax.set_xticks(
            [float(np.mean(v)) for v in boundaries.values()],
            list(boundaries),
            rotation=90,
            fontsize=7,
            color=MUTED,
        )
    _group_legend(
        fig,
        [GROUP6[c] for c in CLASSES],
        loc="upper right",
        bbox_to_anchor=(0.99, 1.0),
        ncol=6,
    )
    fig.suptitle(
        _title(
            f"Mean per recording with one standard deviation, top {TOP_SHIFT} features by "
            "mutual information",
            fast,
        ),
        color=INK,
        fontsize=12,
        x=0.05,
        ha="left",
        y=0.995,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.975))
    path = _save(fig, out_dir / "NB02_recording_shift.png")

    return {
        "per_recording": per_recording,
        "recording_shift": shift,
        "classes_with_many_recordings": multi,
        "classes_with_one_recording": single,
        "figures": [path],
    }

The last question is the one the rest of the project depends on. Take any two of the
nineteen classes. Is there a single measurement that tells them apart.

For every one of the hundred and seventy-one pairs, every feature is scored by the area
under its ROC curve for that pair: the chance that a row drawn from one class scores above
a row drawn from the other, counting a tie as half. One half means the feature is useless
for that pair. One means it separates them completely. The best feature and its score are
kept. Written that way the score comes straight out of the bin counts without sorting
anything, which is what makes seven and a half thousand of these affordable. Rows sharing a
bin count as tied, so the scores are very slightly pessimistic rather than optimistic.

This is deliberately the weakest possible model, one feature and one threshold. A pair that
one column already separates is not going to be the interesting part of a confusion matrix.
A pair that no single column separates is where a real model has to do work, and it is
where the sequence model this project builds later has to earn its place: if a pair cannot
be told apart from one record, the argument for looking at fifty consecutive records is
that the difference is in how the records follow one another rather than in any one of
them.

The matrix is drawn, every pair below the threshold is printed with the feature that came
closest, and the full table of all hundred and seventy-one goes out as a CSV.

In [ ]:
def pairwise_separability(counts, *, fast, out_dir):
    section("Which attacks a single measurement cannot tell apart")

    features = np.array(FEATURES)
    rows = []
    for i, j in itertools.combinations(range(len(CLASSES)), 2):
        auc = eda.pair_auc(counts.counts[i], counts.counts[j])
        filled = np.nan_to_num(auc, nan=-1.0)
        order = np.argsort(-filled)
        best, runner = int(order[0]), int(order[1])
        a, b = CLASSES[i], CLASSES[j]
        rows.append(
            {
                "class_a": a,
                "class_b": b,
                "group_a": GROUP6[a],
                "group_b": GROUP6[b],
                "same_family": GROUP6[a] == GROUP6[b],
                "tier_a": TIER[a],
                "tier_b": TIER[b],
                "rows_a": int(counts.rows_per_class[i]),
                "rows_b": int(counts.rows_per_class[j]),
                "best_feature": features[best],
                "best_auc": float(auc[best]),
                "runner_up_feature": features[runner],
                "runner_up_auc": float(auc[runner]),
                "median_auc_over_features": float(np.nanmedian(auc)),
            }
        )

    table = pd.DataFrame(rows).sort_values("best_auc").reset_index(drop=True)
    n_pairs = len(table)
    hard = table[table["best_auc"] < AUC_THRESHOLD]
    easy = n_pairs - len(hard)

    print(f"pairs                         : {n_pairs}")
    print(f"median best single-feature AUC: {table['best_auc'].median():.4f}")
    print(f"separable at AUC {AUC_THRESHOLD:.2f}        : {easy} of {n_pairs}")
    print(f"below AUC {AUC_THRESHOLD:.2f}               : {len(hard)} of {n_pairs}")
    print()

    print("=" * 79)
    print(f"Pairs no single feature separates at AUC {AUC_THRESHOLD:.2f}")
    print("=" * 79)
    if len(hard):
        print(
            hard[
                ["class_a", "class_b", "best_feature", "best_auc", "runner_up_feature",
                 "runner_up_auc", "same_family", "tier_a", "tier_b"]
            ]
            .assign(
                best_auc=lambda f: f["best_auc"].map(lambda v: f"{v:.4f}"),
                runner_up_auc=lambda f: f["runner_up_auc"].map(lambda v: f"{v:.4f}"),
            )
            .to_string(index=False)
        )
        print()
        worst = table.iloc[0]
        print(f"The least separable pair is {worst['class_a']} against {worst['class_b']}, "
              f"where the")
        print(f"best of the {len(FEATURES)} features is {worst['best_feature']} at "
              f"{worst['best_auc']:.4f}.")
        print()
        involved = pd.concat([hard["class_a"], hard["class_b"]]).value_counts()
        print("classes appearing in the hard pairs, and how often")
        print(involved.to_string())
        print()
        within = int(hard["same_family"].sum())
        print(f"{within} of the {len(hard)} hard pairs are inside a single 6-class family, and")
        print(f"{len(hard) - within} cross families. A pair inside a family is two variants of")
        print("the same attack, which is the case where more context per decision is the")
        print("obvious thing to try.")
    else:
        print(f"None. Every pair is separated at AUC {AUC_THRESHOLD:.2f} by at least one feature.")
    print()

    print("the ten easiest pairs, for contrast")
    print(
        table.tail(10)[["class_a", "class_b", "best_feature", "best_auc"]]
        .assign(best_auc=lambda f: f["best_auc"].map(lambda v: f"{v:.4f}"))
        .to_string(index=False)
    )
    print()

    print("how often each feature is the best one available for a pair")
    winners = table["best_feature"].value_counts()
    print(winners.to_string())

    ordered = sorted(CLASSES, key=lambda c: (GROUP6_ORDER.index(GROUP6[c]), c))
    index = {c: i for i, c in enumerate(ordered)}
    matrix = np.full((len(ordered), len(ordered)), np.nan)
    for _, row in table.iterrows():
        i, j = index[row["class_a"]], index[row["class_b"]]
        matrix[i, j] = matrix[j, i] = row["best_auc"]

    fig, ax = plt.subplots(figsize=(12.5, 11))
    image = ax.imshow(matrix, cmap="RdYlGn", vmin=0.5, vmax=1.0, interpolation="nearest")
    for i in range(len(ordered)):
        for j in range(len(ordered)):
            if i == j or not np.isfinite(matrix[i, j]):
                continue
            hard_cell = matrix[i, j] < AUC_THRESHOLD
            ax.text(
                j,
                i,
                f"{matrix[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=6,
                color=INK,
                fontweight="bold" if hard_cell else "normal",
            )
            if hard_cell:
                ax.add_patch(
                    plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor=INK, linewidth=1.1)
                )
    ax.set_xticks(range(len(ordered)), ordered, rotation=90, fontsize=8, color=MUTED)
    ax.set_yticks(range(len(ordered)), ordered, fontsize=8, color=INK)
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    bar = fig.colorbar(image, ax=ax, fraction=0.035, pad=0.02)
    bar.outline.set_visible(False)
    bar.ax.tick_params(colors=MUTED, length=0)
    bar.set_label("best single-feature AUC", color=MUTED, fontsize=9)
    ax.set_title(
        _title(
            f"Best single feature for each of the {n_pairs} class pairs. "
            f"{len(hard)} outlined pairs fall below {AUC_THRESHOLD:.2f}",
            fast,
        ),
        color=INK,
        fontsize=11,
        loc="left",
        pad=12,
    )
    path = _save(fig, out_dir / "NB02_pairwise_auc.png")

    csv_path = out_dir / "pairwise_auc.csv"
    table.to_csv(csv_path, index=False)
    print(f"wrote {csv_path}   ({len(table)} rows)")

    return {
        "pair_table": table,
        "hard_pairs": hard,
        "pairs_above_threshold": int(easy),
        "pair_csv": csv_path,
        "figures": [path],
    }

Everything a pass computes then goes into one file, so that no later notebook has to redo
any of it and none of them is left to assume it. The document says which mode wrote it, and
the fast pass's copy is marked as not enterable in the ledger, so a sampled number can
never be mistaken for a measured one.

The rankings go in whole rather than truncated to the twenty that were plotted. A later
notebook choosing which features to keep needs the tail as much as the head.

In [ ]:
def eda_document(r, *, git_sha, git_dirty, run_date, seed):
    summary = r["feature_summary"]
    ranking = r["mi_ranking"]
    shift = r["recording_shift"]
    table = r["pair_table"]
    hard = r["hard_pairs"]

    return {
        "generated_by": "AG_PRAXIS_NB02_exploratory_analysis.ipynb",
        "generated_on": run_date,
        "git_sha": git_sha,
        "git_dirty": git_dirty,
        "seed": seed,
        "mode": r["mode"],
        "is_fast_pass": r["fast"],
        "enterable_in_ledger": not r["fast"],
        "source_inventory": str(INVENTORY_PATH),
        "rows_read": r["rows_read"],
        "rows_in_dataset": TOTAL_ROWS_NB01,
        "sampled": r["fast"],
        "sample_rows": r["sample_rows"],
        "sample_rows_per_file_cap": SAMPLE_ROWS_PER_FILE,
        "sample_rows_per_class": r["sample_rows_per_class"],
        "rows_per_class_read": r["rows_per_class_read"],
        "n_features": len(FEATURES),
        "features": FEATURES,
        "dropped_columns": DROPPED,
        "dropped_because": {
            "Drate": "constant across the whole dataset in the NB01 full-scale scan"
        },
        "n_classes": len(CLASSES),
        "classes": CLASSES,
        "bins_per_feature": r["bins_per_feature"],
        "thresholds": {
            "zero_share": ZERO_THRESHOLD,
            "correlation": CORR_THRESHOLD,
            "auc": AUC_THRESHOLD,
        },
        "feature_summary": summary.to_dict(orient="records"),
        "mostly_zero_features": r["mostly_zero"],
        "n_mostly_zero": len(r["mostly_zero"]),
        "correlation": {
            "threshold": CORR_THRESHOLD,
            "computed_on": "the reference sample, identically in both passes",
            "n_pairs_above_threshold": int(len(r["high_correlation_pairs"])),
            "pairs_above_threshold": r["high_correlation_pairs"].to_dict(orient="records"),
            "clustered_order": r["correlation_order"],
            "undefined_for": r["correlation_undefined"],
        },
        "distribution_summary": r["distribution_summary"].to_dict(orient="records"),
        "mutual_information": {
            "units": "nats",
            "estimator": f"exact on the binned feature, up to {NBINS} quantile bins",
            "ceiling_19_class": r["mi_ceiling_19"],
            "ceiling_weakest_4": r["mi_ceiling_weak4"],
            "weakest_4_classes": WEAKEST_CLASSES,
            "weakest_4_rows": r["weak_class_rows"],
            "ranking_19_class": ranking.sort_values("mi_19", ascending=False)[
                ["feature", "mi_19", "rank_19"]
            ].to_dict(orient="records"),
            "ranking_weakest_4": ranking.sort_values("mi_weak4", ascending=False)[
                ["feature", "mi_weak4", "rank_weak4"]
            ].to_dict(orient="records"),
            "largest_rank_changes": ranking.reindex(
                ranking["rank_change"].abs().sort_values(ascending=False).index
            ).head(10)[["feature", "rank_19", "rank_weak4", "rank_change"]].to_dict(
                orient="records"
            ),
        },
        "recording_shift": {
            "statistic": "sqrt(between-recording variance / within-recording variance), "
                         "computed within a class",
            "features": r["mi_ranking"].head(TOP_SHIFT)["feature"].tolist(),
            "classes_with_many_recordings": r["classes_with_many_recordings"],
            "classes_with_one_recording": r["classes_with_one_recording"],
            "per_feature": shift.to_dict(orient="records"),
        },
        "pairwise_separability": {
            "n_pairs": int(len(table)),
            "threshold": AUC_THRESHOLD,
            "n_above_threshold": r["pairs_above_threshold"],
            "n_below_threshold": int(len(hard)),
            "median_best_auc": float(table["best_auc"].median()),
            "least_separable_pair": table.iloc[0][
                ["class_a", "class_b", "best_feature", "best_auc"]
            ].to_dict(),
            "pairs_below_threshold": hard[
                ["class_a", "class_b", "group_a", "group_b", "same_family", "tier_a", "tier_b",
                 "best_feature", "best_auc", "runner_up_feature", "runner_up_auc"]
            ].to_dict(orient="records"),
            "full_table": "pairwise_auc.csv",
        },
        "figures": [p.name for p in r["figures"]],
    }

`run_analysis` calls the steps in order and hands each one what the previous ones produced.
It takes a single argument, and the only thing that argument changes is how many rows the
reading pass reads. Nothing is computed a different way in fast mode.

The reference sample is dropped from the returned result once the steps that need it have
run, so that two passes' worth of it are not sitting in memory at the same time. Everything
computed from it is kept.

In [ ]:
def run_analysis(fast: bool) -> dict:
    mode = "fast" if fast else "full"
    out_dir = OUT_DIRS[mode]
    out_dir.mkdir(parents=True, exist_ok=True)

    started = time.time()
    r = {"mode": mode, "fast": fast, "out_dir": out_dir, "figures": []}

    merge(r, build_reference_sample(ALL_FILES))
    merge(r, build_bin_edges(r["sample"]))
    merge(r, reading_pass(ALL_FILES, r["bin_edges"], r["sample"], fast=fast))
    merge(r, summarise_features(r["moments"], r["counts"], fast=fast, out_dir=out_dir))
    merge(r, correlate_features(r["sample"], fast=fast, out_dir=out_dir))
    merge(r, plot_distributions(r["sample"], fast=fast, out_dir=out_dir))
    merge(r, rank_by_information(r["counts"], fast=fast, out_dir=out_dir))
    merge(
        r,
        measure_recording_shift(r["per_capture"], r["mi_ranking"], fast=fast, out_dir=out_dir),
    )
    merge(r, pairwise_separability(r["counts"], fast=fast, out_dir=out_dir))

    r.pop("sample")

    section(f"Writing the {mode} pass to {out_dir}")
    document = eda_document(r, git_sha=GIT_SHA, git_dirty=GIT_DIRTY, run_date=RUN_DATE, seed=SEED)
    summary_path = out_dir / "eda_summary.json"
    text = json.dumps(cap.jsonable(document), indent=2, default=str) + "\n"
    summary_path.write_text(text)
    print(f"wrote {summary_path}   ({len(text):,} bytes)")
    print(f"wrote {r['pair_csv']}")
    for path in r["figures"]:
        print(f"wrote {path}")
    if IN_COLAB:
        print()
        print("Colab cannot push to the repository from a cell. Download these from Drive and")
        print("move them in from the Mac.")

    r["document"] = document
    r["summary_path"] = summary_path
    r["elapsed_s"] = time.time() - started
    return r

Both passes run here, fast first. If the fast one raises, the full one never starts, which
is the point: a wrong path or a mistake in the arithmetic costs a couple of minutes rather
than the best part of an hour.

In [ ]:
BANNERS = {
    "fast": [
        "FAST PASS",
        "every step, but over a sample, so a test of the code and not of the data",
        "never entered in the ledger",
    ],
    "full": [
        "FULL PASS",
        "every step, every row",
        "this is the pass the repository takes",
    ],
}

results = {}
for fast in (True, False):
    name = "fast" if fast else "full"
    banner(BANNERS[name])
    results[name] = run_analysis(fast)

FAST, FULL = results["fast"], results["full"]
banner([f"fast pass {FAST['elapsed_s']:.0f}s, full pass {FULL['elapsed_s']:.0f}s"])

The full pass's summary file, printed whole. Colab cannot push to the repository from a
cell, so until the artefacts are moved across by hand the saved copy of this notebook is
the only durable record of what the run produced. Printing the file means the executed
notebook carries the result even if the Drive folder is later cleared.

Only the full pass is printed. The fast pass's copies are under `NB02_fast` if they are ever
wanted for debugging, and they are not results.

In [ ]:
print("=" * 79)
print(f"{FULL['summary_path']}   ({FULL['summary_path'].stat().st_size:,} bytes)")
print("=" * 79)
print(FULL["summary_path"].read_text().rstrip())
print()

print("=" * 79)
print(f"{FULL['pair_csv']}   ({FULL['pair_csv'].stat().st_size:,} bytes)")
print("=" * 79)
print(FULL["pair_table"].to_string(index=False))
print()

print("=" * 79)
print("figures, which cannot be printed")
print("=" * 79)
for path in FULL["figures"]:
    print(f"{path.name}   {path.stat().st_size:,} bytes")

Now the two passes side by side.

Only some of what this notebook computes can be compared. The correlation step and the
distribution step both run on the reference sample, which both passes read identically, and
so do the bin edges and everything read off the file names. Those cannot legitimately
differ, so a MISMATCH on any of them is a bug in this notebook rather than something
learned about the dataset, and nothing from a run with a mismatch should be reported.

The other four steps are the point of running twice. The feature summary, the mutual
information rankings, the recording shift and the pairwise AUCs all read a different number
of rows in each pass and are supposed to differ. They are listed underneath with what each
pass read, so the size of the gap between a sampled answer and a measured one is visible
rather than assumed.

In [ ]:
def signature(value, ndigits=10):
    """A comparable form: frames flattened, floats rounded, NaN made equal to itself."""
    if isinstance(value, pd.DataFrame):
        return signature(value.to_dict(orient="records"), ndigits)
    if isinstance(value, pd.Series):
        return signature(value.to_list(), ndigits)
    if isinstance(value, dict):
        return {str(k): signature(v, ndigits) for k, v in value.items()}
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return [signature(v, ndigits) for v in value]
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (float, np.floating)):
        return "nan" if not np.isfinite(value) else round(float(value), ndigits)
    if isinstance(value, (int, np.integer)):
        return int(value)
    return value


COMPARISONS = [
    ("features analysed", lambda r: r["feature_summary"]["feature"].tolist()),
    ("classes seen", lambda r: r["counts"].classes),
    ("reference sample rows", lambda r: r["sample_rows"]),
    ("reference sample per class", lambda r: r["sample_rows_per_class"]),
    ("bins per feature", lambda r: r["bins_per_feature"]),
    ("bin edges", lambda r: {f: signature(list(e)) for f, e in r["bin_edges"].items()}),
    ("correlation matrix", lambda r: signature(r["correlation"])),
    ("feature pairs above 0.95", lambda r: signature(r["high_correlation_pairs"])),
    ("clustered feature order", lambda r: r["correlation_order"]),
    ("correlation undefined for", lambda r: r["correlation_undefined"]),
    ("distribution panels by group", lambda r: signature(r["distribution_summary"])),
    ("recordings found", lambda r: sorted(r["per_capture"])),
    ("classes with many recordings", lambda r: r["classes_with_many_recordings"]),
    ("classes with one recording", lambda r: r["classes_with_one_recording"]),
]

mismatches = []
print(f"{'value':<34} {'result':<10} fast vs full")
print("-" * 79)
for label, get in COMPARISONS:
    a, b = signature(get(FAST)), signature(get(FULL))
    ok = a == b
    print(f"{label:<34} {'PASS' if ok else 'MISMATCH':<10} " + ("" if ok else "see below"))
    if not ok:
        mismatches.append((label, a, b))

print("-" * 79)
print(f"{len(COMPARISONS) - len(mismatches)} of {len(COMPARISONS)} values agree")
COMPARISON_OK = not mismatches

if mismatches:
    print()
    print("The fast path and the full path disagree on something that cannot legitimately")
    print("differ. That is a bug in this notebook, not a finding. Nothing from this run")
    print("should be reported.")
    for label, a, b in mismatches:
        print()
        print(f"  {label}")
        print(f"    fast : {str(a)[:400]}")
        print(f"    full : {str(b)[:400]}")

print()
print("not compared, because these are the steps the two passes are supposed to disagree on")
print(f"  the fast pass read {FAST['rows_read']:,} rows, the full pass {FULL['rows_read']:,}")
print()
print(f"{'step':<34} {'fast':<26} full")
print("-" * 79)
worst_fast = FAST["pair_table"].iloc[0]
worst_full = FULL["pair_table"].iloc[0]
for label, fast_value, full_value in (
    (
        "features zero in >90% of rows",
        f"{len(FAST['mostly_zero'])}",
        f"{len(FULL['mostly_zero'])}",
    ),
    (
        "top feature by mutual information",
        f"{FAST['mi_ranking'].iloc[0]['feature']} {FAST['mi_ranking'].iloc[0]['mi_19']:.4f}",
        f"{FULL['mi_ranking'].iloc[0]['feature']} {FULL['mi_ranking'].iloc[0]['mi_19']:.4f}",
    ),
    (
        "top feature, weakest four classes",
        f"{FAST['mi_ranking'].sort_values('mi_weak4', ascending=False).iloc[0]['feature']}",
        f"{FULL['mi_ranking'].sort_values('mi_weak4', ascending=False).iloc[0]['feature']}",
    ),
    (
        "largest recording shift",
        f"{FAST['recording_shift'].iloc[0]['feature']} "
        f"{FAST['recording_shift'].iloc[0]['max_ratio']:.2f}",
        f"{FULL['recording_shift'].iloc[0]['feature']} "
        f"{FULL['recording_shift'].iloc[0]['max_ratio']:.2f}",
    ),
    (
        "median best pairwise AUC",
        f"{FAST['pair_table']['best_auc'].median():.4f}",
        f"{FULL['pair_table']['best_auc'].median():.4f}",
    ),
    (
        "pairs separable at 0.90",
        f"{FAST['pairs_above_threshold']} of {len(FAST['pair_table'])}",
        f"{FULL['pairs_above_threshold']} of {len(FULL['pair_table'])}",
    ),
    (
        "least separable pair",
        f"{worst_fast['class_a']}/{worst_fast['class_b']} {worst_fast['best_auc']:.4f}",
        f"{worst_full['class_a']}/{worst_full['class_b']} {worst_full['best_auc']:.4f}",
    ),
):
    print(f"{label:<34} {fast_value:<26} {full_value}")
print()
print("Where those two columns are close, the sample happened to be representative for that")
print("quantity. That is worth knowing and it is not a justification for sampling: it is")
print("only visible because the full pass was run.")

The entry for `RESULTS_LEDGER.md`, ready to paste. It reports the full pass only. If the two
passes disagreed on anything they should have agreed on, the entry says so at the top and
the run is not a reference run.

In [ ]:
if not COMPARISON_OK:
    status = "DO NOT ENTER, the fast and full passes disagree and the notebook has a bug"
elif FULL["rows_read"] != TOTAL_ROWS_NB01:
    status = (
        f"DO NOT ENTER, the full pass read {FULL['rows_read']:,} rows and the inventory "
        f"recorded {TOTAL_ROWS_NB01:,}"
    )
elif GIT_DIRTY:
    status = "reference run, working tree dirty"
else:
    status = "reference run"

summary = FULL["feature_summary"]
ranking = FULL["mi_ranking"]
weak_top = ranking.sort_values("mi_weak4", ascending=False).iloc[0]
shift = FULL["recording_shift"].iloc[0]
table, hard = FULL["pair_table"], FULL["hard_pairs"]
worst = table.iloc[0]

ledger = f'''
### NB02 — exploratory analysis ({RUN_DATE})

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB02_exploratory_analysis.ipynb |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| pass reported | full, every step over every row |
| fast/full agreement | {"all {} compared values agree".format(len(COMPARISONS)) if COMPARISON_OK else "MISMATCH, see the comparison cell"} |
| status | {status} |
| runtime | fast {FAST["elapsed_s"]:.0f}s, full {FULL["elapsed_s"]:.0f}s |
| rows read | {FULL["rows_read"]:,} of {TOTAL_ROWS_NB01:,} |
| features analysed | {len(FEATURES)} |
| columns dropped | {", ".join(DROPPED)}, constant across the dataset in NB01 |
| features zero in more than {ZERO_THRESHOLD:.0%} of rows | {len(FULL["mostly_zero"])} of {len(FEATURES)} |
| most skewed feature | {summary.loc[summary["skew"].idxmax(), "feature"]}, skew {summary["skew"].max():,.1f} |
| feature pairs above r {CORR_THRESHOLD} | {len(FULL["high_correlation_pairs"])} |
| top feature by mutual information, {len(CLASSES)}-class | {ranking.iloc[0]["feature"]}, {ranking.iloc[0]["mi_19"]:.4f} nats of {FULL["mi_ceiling_19"]:.4f} |
| top feature, four weakest classes | {weak_top["feature"]}, {weak_top["mi_weak4"]:.4f} nats of {FULL["mi_ceiling_weak4"]:.4f} |
| largest between-recording shift | {shift["feature"]}, ratio {shift["max_ratio"]:.2f} on {shift["worst_class"]} |
| median best single-feature AUC, {len(table)} pairs | {table["best_auc"].median():.4f} |
| pairs separable at AUC {AUC_THRESHOLD:.2f} | {FULL["pairs_above_threshold"]} of {len(table)} |
| pairs below AUC {AUC_THRESHOLD:.2f} | {len(hard)} of {len(table)} |
| least separable pair | {worst["class_a"]} / {worst["class_b"]}, {worst["best_auc"]:.4f} on {worst["best_feature"]} |
| metrics | none, no model trained |
| artefacts | {FULL["out_dir"]}, holding eda_summary.json, pairwise_auc.csv and {len(FULL["figures"])} figures |

Features zero in more than {ZERO_THRESHOLD:.0%} of rows: {", ".join(FULL["mostly_zero"]) or "none"}

Pairs below AUC {AUC_THRESHOLD:.2f}: {" · ".join(f"{r.class_a}/{r.class_b} {r.best_auc:.4f} on {r.best_feature}" for r in hard.itertuples()) or "none"}
'''

print(ledger)